# Research agent — Browserless × LlamaIndex (Python)

Multi-step research workflow using Browserless's stateless tools: search the web with `browserless_search`, then scrape the most relevant results with `browserless_smartscraper`, then summarize. The LLM picks which tool to call at each step.

All tools used here are stateless single-shot calls — no session affinity required.

In [ ]:
%pip install -q llama-index llama-index-tools-mcp llama-index-llms-anthropic

In [ ]:
import os
from llama_index.tools.mcp import BasicMCPClient, McpToolSpec
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.anthropic import Anthropic

client = BasicMCPClient(
    "https://mcp.browserless.io/mcp",
    headers={"Authorization": f"Bearer {os.environ['BROWSERLESS_TOKEN']}"},
    timeout=120,
)

tools = await McpToolSpec(client=client).to_tool_list_async()
stateless_tools = [
    t for t in tools
    if t.metadata.name not in ("browserless_agent", "browserless_skill")
]

agent = FunctionAgent(
    name="BrowserlessResearchAgent",
    description="Researches topics by searching the web and scraping the most relevant pages.",
    llm=Anthropic(model="claude-sonnet-4-6"),
    tools=stateless_tools,
    system_prompt="You are a research assistant. Use browserless_search to find sources, then browserless_smartscraper to read them. Cite URLs in your output.",
)

In [ ]:
prompt = (
    "Research the current state of WebAssembly outside the browser. "
    "Use browserless_search to find 3 recent authoritative articles, then use browserless_smartscraper "
    "to read the most relevant one and summarize the key takeaways in 5 bullet points. "
    "Cite the URL for each takeaway."
)

response = await agent.run(prompt)
print(response)